# Apuntes completos — Notebook 07: PyTorch Experiment Tracking

Este cuaderno resume de punta a punta el notebook 07 del curso *Learn PyTorch for Deep Learning*.

**Objetivo práctico:** diseñar, ejecutar y comparar múltiples experimentos de clasificación de imágenes (pizza, steak, sushi) utilizando modelos preentrenados, y registrar los resultados de forma sistemática con TensorBoard.

## 1. ¿Por qué rastrear experimentos?

En machine learning, es habitual probar decenas de combinaciones de hiperparámetros, arquitecturas y datos. Sin un sistema de seguimiento, las comparaciones se vuelven inmanejables.

**Experiment tracking** resuelve esto: registra métricas, hiperparámetros y artefactos de cada experimento para facilitar la comparación y la reproducibilidad.

### Herramientas más comunes

| Herramienta | Pros | Contras | Coste |
|---|---|---|---|
| Diccionarios / CSV | Sencillo, puro Python | No escala | Gratis |
| **TensorBoard** | Integración nativa con PyTorch, ampliamente usado | UX menos pulida | Gratis |
| Weights & Biases | UX excelente, compartir resultados | Dependencia externa | Gratis uso personal |
| MLFlow | Open source, ciclo MLOps completo | Setup servidor remoto más complejo | Gratis |

En este notebook se utiliza **TensorBoard** por su integración directa con `torch.utils.tensorboard.SummaryWriter`.

## 2. Contenidos del notebook

| Sección | Descripción |
|---|---|
| 3. Setup | Importaciones, seeds y device |
| 4. Datos | Descarga de datasets (10% y 20%) |
| 5. DataLoaders | Transforms manuales vs automáticas, creación de DataLoaders |
| 6. Modelo | Modelo preentrenado EfficientNet como feature extractor |
| 7. Entrenamiento con tracking | Función `train()` modificada con `SummaryWriter` |
| 8. Visualización en TensorBoard | Cómo ver y lanzar TensorBoard |
| 9. Helper `create_writer()` | Función auxiliar para organizar logs por experimento |
| 10. Batería de experimentos | Bucle combinatorio: 2 datasets × 2 modelos × 2 duraciones |
| 11. Análisis de resultados | Tabla comparativa y conclusiones |
| 12. Predicciones del mejor modelo | Carga y evaluación visual |
| 13. Buenas prácticas | Checklist de experiment tracking |

## 3. Setup

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import sys

# Permitir imports del directorio raíz del proyecto
sys.path.append('..')

from torch import nn
from torchvision import transforms
from torchinfo import summary
from torch.utils.tensorboard import SummaryWriter

assert int(torch.__version__.split('.')[1]) >= 12, 'Se requiere torch >= 1.12'
assert int(torchvision.__version__.split('.')[1]) >= 13, 'Se requiere torchvision >= 0.13'

print(f'torch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
def set_seeds(seed: int = 42) -> None:
    """Fija las semillas para reproducibilidad en CPU y CUDA.

    Args:
        seed: Valor de la semilla. Por defecto 42.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

## 4. Descarga de datos

Se usan dos subconjuntos del dataset Food101 (solo pizza, steak, sushi):
- **10%** del training set original.
- **20%** del training set original.

Ambos comparten el **mismo test set** para que las comparaciones sean válidas.

In [ ]:
import os
import zipfile
import requests
from pathlib import Path


def download_data(source: str,
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Descarga un dataset comprimido y lo extrae.

    Args:
        source: URL del archivo .zip.
        destination: Nombre del subdirectorio destino dentro de data/.
        remove_source: Si True, elimina el .zip tras la extracción.

    Returns:
        Ruta al directorio con los datos extraídos.
    """
    data_path = Path('../data/')
    image_path = data_path / destination

    if image_path.is_dir():
        print(f'[INFO] {image_path} ya existe, omitiendo descarga.')
    else:
        print(f'[INFO] Creando {image_path}...')
        image_path.mkdir(parents=True, exist_ok=True)

        target_file = Path(source).name
        with open(data_path / target_file, 'wb') as f:
            request = requests.get(source)
            print(f'[INFO] Descargando {target_file}...')
            f.write(request.content)

        with zipfile.ZipFile(data_path / target_file, 'r') as zip_ref:
            print(f'[INFO] Descomprimiendo {target_file}...')
            zip_ref.extractall(image_path)

        if remove_source:
            os.remove(data_path / target_file)

    return image_path


data_10_percent_path = download_data(
    source='https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip',
    destination='pizza_steak_sushi'
)

data_20_percent_path = download_data(
    source='https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip',
    destination='pizza_steak_sushi_20_percent'
)

In [ ]:
# Rutas de entrenamiento y test
train_dir_10_percent = data_10_percent_path / 'train'
train_dir_20_percent = data_20_percent_path / 'train'
test_dir = data_10_percent_path / 'test'  # mismo test para todos

print(f'Train 10%: {train_dir_10_percent}')
print(f'Train 20%: {train_dir_20_percent}')
print(f'Test:      {test_dir}')

## 5. Transforms y DataLoaders

### Transforms manuales vs automáticas

- **Manual:** `Resize` → `ToTensor` → `Normalize` (con estadísticas de ImageNet).
- **Automática:** `weights.transforms()` — extrae el pipeline exacto usado durante el preentrenamiento.

La opción automática garantiza consistencia con el modelo preentrenado. La opción manual ofrece más control.

Para los experimentos se usa la transform **manual** normalizada con estadísticas de ImageNet.

In [ ]:
# Normalización con estadísticas de ImageNet
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

# Comparar con transforms automáticas
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
auto_transforms = weights.transforms()

print(f'Transform manual: {simple_transform}')
print(f'\nTransform automática: {auto_transforms}')

In [ ]:
from going_modular.going_modular import data_setup

BATCH_SIZE = 32

# DataLoaders con 10% de datos
train_dataloader_10, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir_10_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

# DataLoaders con 20% de datos
train_dataloader_20, _, _ = data_setup.create_dataloaders(
    train_dir=train_dir_20_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

print(f'Batches train 10%: {len(train_dataloader_10)}')
print(f'Batches train 20%: {len(train_dataloader_20)}')
print(f'Batches test:      {len(test_dataloader)}')
print(f'Clases: {class_names}')

## 6. Creación del modelo (feature extractor)

Se utiliza **transfer learning** con la estrategia de **feature extraction**:
1. Cargar modelo preentrenado (EfficientNet_B0 o B2) con pesos de ImageNet.
2. Congelar las capas base (`model.features`).
3. Reemplazar la cabeza clasificadora para 3 clases.

### Parámetros de los modelos

| Modelo | Parámetros totales | Parámetros entrenables | `in_features` clasificador |
|---|---|---|---|
| EfficientNet_B0 | ~5.3M | ~3,843 | 1280 |
| EfficientNet_B2 | ~9.1M | ~4,227 | 1408 |

In [ ]:
OUT_FEATURES = len(class_names)


def create_effnetb0() -> torch.nn.Module:
    """Crea un EfficientNet_B0 como feature extractor para 3 clases."""
    weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
    model = torchvision.models.efficientnet_b0(weights=weights).to(device)

    for param in model.features.parameters():
        param.requires_grad = False

    set_seeds()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features=1280, out_features=OUT_FEATURES)
    ).to(device)

    model.name = 'effnetb0'
    print(f'[INFO] Modelo {model.name} creado.')
    return model


def create_effnetb2() -> torch.nn.Module:
    """Crea un EfficientNet_B2 como feature extractor para 3 clases."""
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    model = torchvision.models.efficientnet_b2(weights=weights).to(device)

    for param in model.features.parameters():
        param.requires_grad = False

    set_seeds()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features=1408, out_features=OUT_FEATURES)
    ).to(device)

    model.name = 'effnetb2'
    print(f'[INFO] Modelo {model.name} creado.')
    return model


# Verificación rápida
effnetb0 = create_effnetb0()
effnetb2 = create_effnetb2()

## 7. Función de entrenamiento con tracking

Se modifica la función `train()` de `engine.py` para aceptar un parámetro `writer` (`SummaryWriter`). En cada época se registran:

- `Loss/train_loss` y `Loss/test_loss`
- `Accuracy/train_acc` y `Accuracy/test_acc`

Esto permite visualizar curvas de aprendizaje directamente en TensorBoard.

In [ ]:
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from going_modular.going_modular.engine import train_step, test_step


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer: Optional[SummaryWriter] = None
          ) -> Dict[str, List]:
    """Entrena y evalúa un modelo PyTorch, registrando métricas en TensorBoard.

    Args:
        model: Modelo PyTorch a entrenar.
        train_dataloader: DataLoader de entrenamiento.
        test_dataloader: DataLoader de test.
        optimizer: Optimizador.
        loss_fn: Función de pérdida.
        epochs: Número de épocas.
        device: Dispositivo de cómputo.
        writer: Instancia de SummaryWriter (opcional).

    Returns:
        Diccionario con listas de métricas por época.
    """
    results: Dict[str, List] = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': []
    }

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(
            model=model, dataloader=train_dataloader,
            loss_fn=loss_fn, optimizer=optimizer, device=device
        )
        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader,
            loss_fn=loss_fn, device=device
        )

        print(
            f'Epoch: {epoch+1} | '
            f'train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | '
            f'test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}'
        )

        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)

        # Registrar en TensorBoard si hay writer
        if writer:
            writer.add_scalars(
                main_tag='Loss',
                tag_scalar_dict={'train_loss': train_loss, 'test_loss': test_loss},
                global_step=epoch
            )
            writer.add_scalars(
                main_tag='Accuracy',
                tag_scalar_dict={'train_acc': train_acc, 'test_acc': test_acc},
                global_step=epoch
            )
            writer.close()

    return results

## 8. Visualización con TensorBoard

TensorBoard lee los archivos generados por `SummaryWriter` en el directorio `runs/`.

### Cómo lanzar TensorBoard

| Entorno | Comando |
|---|---|
| Jupyter / Colab | `%load_ext tensorboard` → `%tensorboard --logdir runs` |
| VS Code | `Shift+Ctrl+P` → *Python: Launch TensorBoard* |
| Terminal | `tensorboard --logdir runs` |

In [ ]:
# Descomentar para visualizar TensorBoard en el notebook
# %load_ext tensorboard
# %tensorboard --logdir ../runs

## 9. Helper `create_writer()`

Para organizar los logs de múltiples experimentos, se crea una función que genera un `SummaryWriter` con una ruta estructurada:

```
runs/YYYY-MM-DD/experiment_name/model_name/extra
```

Esto permite filtrar fácilmente por fecha, dataset, modelo y número de épocas en TensorBoard.

In [ ]:
from datetime import datetime


def create_writer(experiment_name: str,
                  model_name: str,
                  extra: Optional[str] = None) -> SummaryWriter:
    """Crea un SummaryWriter con log_dir estructurado.

    Args:
        experiment_name: Nombre del experimento (ej. 'data_10_percent').
        model_name: Nombre del modelo (ej. 'effnetb0').
        extra: Información adicional (ej. '5_epochs').

    Returns:
        Instancia de SummaryWriter.
    """
    timestamp = datetime.now().strftime('%Y-%m-%d')

    if extra:
        log_dir = os.path.join('..', 'runs', timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join('..', 'runs', timestamp, experiment_name, model_name)

    print(f'[INFO] SummaryWriter creado en: {log_dir}')
    return SummaryWriter(log_dir=log_dir)


# Ejemplo
example_writer = create_writer(
    experiment_name='data_10_percent',
    model_name='effnetb0',
    extra='5_epochs'
)

## 10. Experimento de demostración

Antes de lanzar la batería completa de 8 experimentos, se ejecuta **uno solo** para verificar que el pipeline funciona correctamente: EfficientNet_B0 con 10% de datos durante 5 épocas.

In [ ]:
# Crear modelo fresco
set_seeds()
demo_model = create_effnetb0()

# Loss y optimizador
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(demo_model.parameters(), lr=0.001)

# Entrenar con tracking
demo_writer = create_writer(
    experiment_name='data_10_percent',
    model_name='effnetb0',
    extra='5_epochs_demo'
)

set_seeds()
demo_results = train(
    model=demo_model,
    train_dataloader=train_dataloader_10,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device,
    writer=demo_writer
)

demo_results

## 11. Batería completa de 8 experimentos (referencia)

El diseño experimental combina tres ejes:

| Eje | Valores |
|---|---|
| Datos de entrenamiento | 10% , 20% |
| Modelo | EfficientNet_B0, EfficientNet_B2 |
| Épocas | 5, 10 |

Esto produce **2 × 2 × 2 = 8 combinaciones**:

| # | Dataset | Modelo | Épocas |
|---|---|---|---|
| 1 | 10% | EffNetB0 | 5 |
| 2 | 10% | EffNetB2 | 5 |
| 3 | 10% | EffNetB0 | 10 |
| 4 | 10% | EffNetB2 | 10 |
| 5 | 20% | EffNetB0 | 5 |
| 6 | 20% | EffNetB2 | 5 |
| 7 | 20% | EffNetB0 | 10 |
| 8 | 20% | EffNetB2 | 10 |

**Nota:** El siguiente bloque de código está comentado. Para ejecutarlo, descoméntalo (tiempo estimado: ~38 min en GPU).

In [ ]:
# =====================================================================
# CÓDIGO DEL BUCLE COMPLETO DE EXPERIMENTOS (descomentar para ejecutar)
# =====================================================================

# from going_modular.going_modular.utils import save_model
#
# # Configuración de experimentos
# num_epochs = [5, 10]
# models = ['effnetb0', 'effnetb2']
# train_dataloaders = {
#     'data_10_percent': train_dataloader_10,
#     'data_20_percent': train_dataloader_20
# }
#
# set_seeds(seed=42)
# experiment_number = 0
#
# for dataloader_name, train_dataloader in train_dataloaders.items():
#     for epochs in num_epochs:
#         for model_name in models:
#             experiment_number += 1
#             print(f'\n[INFO] Experimento {experiment_number}')
#             print(f'[INFO] Modelo: {model_name}')
#             print(f'[INFO] Datos: {dataloader_name}')
#             print(f'[INFO] Épocas: {epochs}')
#
#             # Crear modelo nuevo para cada experimento
#             if model_name == 'effnetb0':
#                 model = create_effnetb0()
#             else:
#                 model = create_effnetb2()
#
#             loss_fn = nn.CrossEntropyLoss()
#             optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
#
#             train(
#                 model=model,
#                 train_dataloader=train_dataloader,
#                 test_dataloader=test_dataloader,
#                 optimizer=optimizer,
#                 loss_fn=loss_fn,
#                 epochs=epochs,
#                 device=device,
#                 writer=create_writer(
#                     experiment_name=dataloader_name,
#                     model_name=model_name,
#                     extra=f'{epochs}_epochs'
#                 )
#             )
#
#             # Guardar modelo
#             save_filepath = f'07_{model_name}_{dataloader_name}_{epochs}_epochs.pth'
#             save_model(
#                 model=model,
#                 target_dir='../models',
#                 model_name=save_filepath
#             )
#             print('-' * 50)

## 12. Análisis de resultados

Los resultados obtenidos en el notebook original muestran las siguientes tendencias:

### Tabla resumen (resultados de referencia)

| # | Modelo | Datos | Épocas | Test Loss (final) | Test Acc (final) |
|---|---|---|---|---|---|
| 1 | EffNetB0 | 10% | 5 | 0.568 | 88.6% |
| 2 | EffNetB2 | 10% | 5 | 0.708 | 88.7% |
| 3 | EffNetB0 | 10% | 10 | 0.464 | 90.7% |
| 4 | EffNetB2 | 10% | 10 | 0.587 | 88.7% |
| 5 | EffNetB0 | 20% | 5 | 0.391 | 91.8% |
| 6 | EffNetB2 | 20% | 5 | 0.446 | 94.9% |
| 7 | EffNetB0 | 20% | 10 | 0.278 | 90.7% |
| 8 | **EffNetB2** | **20%** | **10** | **0.391** | **93.8%** |

### Observaciones clave

1. **Más datos mejoran el rendimiento.** Pasar de 10% a 20% de datos produce mejoras consistentes tanto en loss como en accuracy.

2. **Más parámetros pueden ayudar.** EffNetB2 (9.1M params) tiende a superar a EffNetB0 (5.3M params), especialmente con más datos.

3. **Más épocas no siempre ayudan proporcionalmente.** El modelo 6 (EffNetB2, 20%, 5 épocas) logra resultados similares al modelo 8 (misma configuración, 10 épocas), sugiriendo rendimientos decrecientes.

4. **The Bitter Lesson** (Rich Sutton, 2019): los factores que más escalan con más computación (más datos, modelos más grandes) tienden a dominar a largo plazo frente a la ingeniería manual de características.

5. **Para un primer acercamiento, empezar pequeño.** Los primeros experimentos deben ser rápidos (segundos a minutos). Solo se escala lo que funciona.

## 13. Predicciones con el mejor modelo

El modelo con mejor rendimiento global (experimento 8: EffNetB2, 20% datos, 10 épocas) se guarda en:

```
models/07_effnetb2_data_20_percent_10_epochs.pth
```

Para cargarlo: crear una instancia nueva del modelo → cargar `state_dict` → evaluar.

In [ ]:
# Cargar el mejor modelo (descomentar si se ejecutó el bucle completo)
# best_model_path = '../models/07_effnetb2_data_20_percent_10_epochs.pth'
# best_model = create_effnetb2()
# best_model.load_state_dict(torch.load(best_model_path))
#
# # Tamaño del modelo en disco
# model_size_mb = Path(best_model_path).stat().st_size // (1024 * 1024)
# print(f'Tamaño del modelo: {model_size_mb} MB')

In [ ]:
import random
from PIL import Image


def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: list,
                        transform: transforms.Compose,
                        device: torch.device) -> None:
    """Predice la clase de una imagen y la visualiza.

    Args:
        model: Modelo entrenado.
        image_path: Ruta a la imagen.
        class_names: Lista de nombres de clases.
        transform: Pipeline de transformación.
        device: Dispositivo de cómputo.
    """
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.inference_mode():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    pred_idx = int(torch.argmax(probs, dim=1).item())
    pred_class = class_names[pred_idx]
    pred_prob = float(torch.max(probs).item())

    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f'Pred: {pred_class} | Prob: {pred_prob:.3f}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Ejemplo de predicción con imágenes del test set
# (descomentar si se cargó el mejor modelo arriba)
#
# import random
# test_images = list(Path(test_dir).rglob('*.jpg'))
# sample_images = random.sample(test_images, k=min(3, len(test_images)))
#
# for img_path in sample_images:
#     pred_and_plot_image(
#         model=best_model,
#         image_path=str(img_path),
#         class_names=class_names,
#         transform=simple_transform,
#         device=device
#     )

## 14. Checklist de buenas prácticas en experiment tracking

- **Reproducibilidad:** fijar semillas con `set_seeds()` antes de cada experimento.
- **Mismo test set:** todos los experimentos deben evaluarse sobre el mismo conjunto de test.
- **Modelo nuevo por experimento:** instanciar un modelo fresco para que todos partan del mismo punto.
- **Log_dir organizado:** usar una estructura de directorios legible (`fecha/dataset/modelo/épocas`).
- **Guardar modelos:** persistir los pesos de cada experimento para poder recuperar el mejor.
- **Empezar pequeño:** experimentos iniciales rápidos → escalar solo lo que funcione.
- **Registrar hiperparámetros:** learning rate, batch size, épocas, arquitectura.
- **Cerrar writers:** llamar a `writer.close()` al finalizar para asegurar que los datos se escriben.
- **Consistencia de transforms:** usar el mismo pipeline de preprocesamiento en train, test e inferencia.

## 15. Hiperparámetros del notebook

| Hiperparámetro | Valor |
|---|---|
| Arquitecturas | EfficientNet_B0, EfficientNet_B2 |
| Pesos | DEFAULT (mejores disponibles) |
| Input size | 224 × 224 |
| Batch size | 32 |
| Épocas | 5, 10 |
| Optimizer | Adam |
| Learning rate | 0.001 |
| Loss | CrossEntropyLoss |
| Dropout B0 | 0.2 |
| Dropout B2 | 0.3 |
| Clases | 3 (pizza, steak, sushi) |
| Seed | 42 |

## 16. Conclusiones

1. **Experiment tracking es esencial** cuando se prueban múltiples combinaciones. TensorBoard proporciona una solución integrada con PyTorch que escala bien.

2. **`SummaryWriter`** es la clase central: registra escalares, grafos del modelo y otros artefactos en formato TensorBoard.

3. **El diseño experimental importa tanto como el código.** Definir claramente qué se varía (datos, modelo, épocas) y qué se mantiene constante (test set, transforms, seeds) es fundamental para conclusiones válidas.

4. **Tendencia general:** más datos y modelos más grandes producen mejores resultados, pero con rendimientos decrecientes. Esto es consistente con *The Bitter Lesson*.

5. **Para el TFG:** este workflow (diseño experimental → ejecución automatizada → comparación en TensorBoard) es directamente aplicable a cualquier proyecto de clasificación.